# SVD Quantization on GPU

Run SVD-based sub-1-bit quantization with GPU acceleration.

**Runtime**: Runtime > Change runtime type > **GPU** (T4 or better)

**Expected time**: ~30-60 minutes depending on model size

In [11]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [15]:
# Clone/pull the repo
!git clone https://github.com/toxzak-svg/Quantization-Exploration.git /content/quantization-exploration 2>/dev/null || (cd /content/quantization-exploration && git pull)
%cd /content/quantization-exploration
!git pull origin main

Already up to date.
/content/quantization-exploration
From https://github.com/toxzak-svg/Quantization-Exploration
 * branch            main       -> FETCH_HEAD
Already up to date.


In [16]:
# Install dependencies
!pip install torch transformers accelerate numpy scikit-learn -q

In [17]:
# Download Gemma 2 2B model
from huggingface_hub import snapshot_download
from google.colab import userdata
from safetensors.torch import load_file, save_file
import os
import torch

token = userdata.get('HF_TOKEN')
MODEL_DIR = "/content/models/gemma-4-E2B"

snapshot_download(
    repo_id="google/gemma-4-E2B-it",
    local_dir=MODEL_DIR,
    token=token
)

# Merge shards if script expects a single model.safetensors
target_path = os.path.join(MODEL_DIR, "model.safetensors")
if not os.path.exists(target_path):
    print("Merging sharded safetensors into single file...")
    shards = sorted([f for f in os.listdir(MODEL_DIR) if f.startswith("model-") and f.endswith(".safetensors")])
    combined_state_dict = {}
    for shard in shards:
        combined_state_dict.update(load_file(os.path.join(MODEL_DIR, shard)))
    save_file(combined_state_dict, target_path)
    print("Successfully created model.safetensors")

print(f"Files in {MODEL_DIR}:")
print(os.listdir(MODEL_DIR))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Files in /content/models/gemma-4-E2B:
['generation_config.json', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model', 'README.md', 'model.safetensors', 'model-00001-of-00002.safetensors', '.gitattributes', '.cache', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json']


In [31]:
import os

script_path = 'scripts/quantize_svd_proper_v2.py'
with open(script_path, 'r') as f:
    content = f.read()

# Remove the restrictive 'language_model' check and simplify weight detection
# We'll target the loop that filters keys
old_logic = """        if 'language_model' not in key:
            continue"""
content = content.replace(old_logic, "        # Removed language_model restriction")

# Also broaden the exclusion list to ensure we don't accidentally skip layers
content = content.replace("if any(x in key for x in ['lm_head', 'embed_tokens', 'norm', 'audio_tower', 'vision_tower', 'embed_vision']):",
                          "if any(x in key for x in ['embed_tokens', 'norm', 'audio_tower', 'vision_tower', 'embed_vision']):")

# Ensure we don't crash on ZeroDivisionError if it somehow still finds nothing
content = content.replace("stats['compression'] = stats['total_original'] * 16 / stats['total_bits']",
                          "stats['compression'] = (stats['total_original'] * 16 / stats['total_bits']) if stats['total_bits'] > 0 else 0")

with open(script_path, 'w') as f:
    f.write(content)

print("Patched script to support Gemma-2 layer naming (removed 'language_model' requirement).")

# Attempt execution again
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_60.pt \
    --threshold 0.60

Patched script to support Gemma-2 layer naming (removed 'language_model' requirement).
Found 182 weights
Energy threshold: 0.6 (60%)
Layer 0: rank= 802 (35% of full), bpw=0.4352, shape=[2304,9216]
Layer 1: rank= 814 (35% of full), bpw=0.4417, shape=[9216,2304]
Layer 2: rank= 856 (37% of full), bpw=0.4645, shape=[9216,2304]
Layer 3: rank= 281 (27% of full), bpw=0.3966, shape=[1024,2304]
Layer 4: rank= 370 (18% of full), bpw=0.3414, shape=[2304,2048]
Layer 50: rank= 778 (34% of full), bpw=0.4222, shape=[9216,2304]
Layer 100: rank= 845 (37% of full), bpw=0.4585, shape=[9216,2304]
Layer 150: rank= 283 (28% of full), bpw=0.3995, shape=[1024,2304]

Results:
  Avg rank: 551
  Avg bits/weight: 0.4356
  Compression: 36.7x

Saved to quantized/gemma_svd_60.pt


In [32]:
import os
import subprocess

# 1. Reset file to clear out any corrupted patches from previous runs
subprocess.run(["git", "restore", "scripts/eval_reconstruction.py"], cwd="/content/quantization-exploration")

eval_path = 'scripts/eval_reconstruction.py'
with open(eval_path, 'r') as f:
    content = f.read()

svd_helper = """
def reconstruct_from_svd(q_entry, device='cpu'):
    U = q_entry['U'].to(device)
    S = q_entry['S'].to(device)
    Vt = q_entry['Vt'].to(device)
    return torch.matmul(U * S, Vt)
"""

# 2. Precisely replace the target logic with exact indentation matching the original file (8 spaces)
target_line = "        W_rec = reconstruct_fn(q_entry, 'cpu')"
new_logic = """        if isinstance(q_entry, dict) and 'U' in q_entry and 'Vt' in q_entry:
            W_rec = reconstruct_from_svd(q_entry, 'cpu')
        else:
            W_rec = reconstruct_fn(q_entry, 'cpu')"""

if target_line in content:
    content = content.replace(target_line, new_logic)
    content = content.replace("def main():", svd_helper + "\ndef main():")

    with open(eval_path, 'w') as f:
        f.write(content)
    print('Successfully reset and cleanly patched eval_reconstruction.py for SVD support.')
else:
    print('Could not find the target line to patch. The file might be structured differently than expected.')

Successfully reset and cleanly patched eval_reconstruction.py for SVD support.


In [33]:
# Run 70% threshold for comparison
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_70.pt \
    --threshold 0.70

Found 182 weights
Energy threshold: 0.7 (70%)
Layer 0: rank=1035 (45% of full), bpw=0.5616, shape=[2304,9216]
Layer 1: rank=1047 (45% of full), bpw=0.5681, shape=[9216,2304]
Layer 2: rank=1086 (47% of full), bpw=0.5893, shape=[9216,2304]
Layer 3: rank= 370 (36% of full), bpw=0.5222, shape=[1024,2304]
Layer 4: rank= 507 (25% of full), bpw=0.4678, shape=[2304,2048]
Layer 50: rank=1008 (44% of full), bpw=0.5470, shape=[9216,2304]
Layer 100: rank=1069 (46% of full), bpw=0.5801, shape=[9216,2304]
Layer 150: rank= 371 (36% of full), bpw=0.5237, shape=[1024,2304]

Results:
  Avg rank: 712
  Avg bits/weight: 0.5595
  Compression: 28.6x

Saved to quantized/gemma_svd_70.pt


In [37]:
# Rerun evaluation with the patched reconstruction logic for both files
print("Evaluating 60% Energy Threshold:")
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --quantized quantized/gemma_svd_60.pt \
    --max-layers 10

print("\nEvaluating 70% Energy Threshold:")
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --quantized quantized/gemma_svd_70.pt \
    --max-layers 10

Evaluating 60% Energy Threshold:
usage: eval_reconstruction.py [-h] [--model-dir MODEL_DIR]
                              [--max-layers MAX_LAYERS]
eval_reconstruction.py: error: unrecognized arguments: --quantized quantized/gemma_svd_60.pt

Evaluating 70% Energy Threshold:
usage: eval_reconstruction.py [-h] [--model-dir MODEL_DIR]
                              [--max-layers MAX_LAYERS]
eval_reconstruction.py: error: unrecognized arguments: --quantized quantized/gemma_svd_70.pt


In [59]:
import shutil
import os
from google.colab import drive

# Mount drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the destination folder in Drive
drive_results_path = "/content/drive/MyDrive/quantization-results"
os.makedirs(drive_results_path, exist_ok=True)

# List of model files we've generated
models_to_copy = [
    "quantized/gemma_svd_60.pt",
    "quantized/gemma_svd_70.pt"
]

for f in models_to_copy:
    if os.path.exists(f):
        dest = os.path.join(drive_results_path, os.path.basename(f))
        shutil.copy(f, dest)
        print(f"Successfully saved {f} to {dest}")
    else:
        print(f"File {f} not found locally.")

Successfully saved quantized/gemma_svd_60.pt to /content/drive/MyDrive/quantization-results/gemma_svd_60.pt
Successfully saved quantized/gemma_svd_70.pt to /content/drive/MyDrive/quantization-results/gemma_svd_70.pt


In [ ]:
import os

eval_path = 'scripts/eval_reconstruction.py'

# Full script for evaluation
full_script = """
import torch
import numpy as np
from pathlib import Path
from typing import Dict, List
import argparse
from safetensors.torch import load_file
import math
import os

def load_gemma_weights_slice(model_dir: str):
    model_dir = Path(model_dir)
    safetensor_path = model_dir / 'model.safetensors'
    weights = load_file(safetensor_path)
    filtered = {}
    for k, v in weights.items():
        if 'weight' in k and len(v.shape) == 2 and not any(n in k for n in ['norm', 'embed']):
            filtered[k] = v
    return filtered

def unpack_ternary(packed, shape, device='cpu'):
    packed = packed.to(torch.int32).to(device)
    powers = torch.tensor([81, 27, 9, 3, 1], dtype=torch.int32, device=device)
    encoded = (packed.unsqueeze(1) // powers) % 3
    encoded = encoded.flatten()
    n = math.prod(shape)
    t = encoded[:n].to(torch.float32) - 1.0
    return t.reshape(shape)

def reconstruct_from_svd(q_entry, device='cpu'):
    if 'U_packed' in q_entry:
        U = unpack_ternary(q_entry['U_packed'], q_entry['U_shape'], device)
        Vt = unpack_ternary(q_entry['Vt_packed'], q_entry['Vt_shape'], device)
        S = q_entry['S'].to(device).to(torch.float32)
        if 'U_scale' in q_entry: U = U * float(q_entry['U_scale'])
        if 'Vt_scale' in q_entry: Vt = Vt * float(q_entry['Vt_scale'])
        if 'S_scale' in q_entry: S = S * float(q_entry['S_scale'])
        return torch.matmul(U * S, Vt)
    norm_entry = {k.lower(): v for k, v in q_entry.items() if isinstance(v, torch.Tensor)}
    U, S, Vt = norm_entry.get('u'), norm_entry.get('s'), norm_entry.get('vt')
    if all(x is not None for x in [U, S, Vt]):
        return torch.matmul(U.to(device).to(torch.float32) * S.to(device).to(torch.float32), Vt.to(device).to(torch.float32))
    return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-dir', default='models/gemma-4-E2B')
    parser.add_argument('--quantized', type=str, required=True)
    args = parser.parse_args()

    orig_weights = load_gemma_weights_slice(args.model_dir)
    sorted_orig_keys = sorted(orig_weights.keys())

    q_data = torch.load(args.quantized, weights_only=False)
    if 'quantized' in q_data: q_data = q_data['quantized']
    lookup = {int(k): v for k, v in q_data.items() if str(k).isdigit()}

    mses = []
    for i, model_key in enumerate(sorted_orig_keys):
        if i in lookup:
            W_rec = reconstruct_from_svd(lookup[i])
            if W_rec is not None:
                W_orig = orig_weights[model_key].to(torch.float32)
                mse = torch.mean((W_orig - W_rec.to(torch.float32))**2).item()
                mses.append(mse)

    if mses:
        print(f"\\nResults for {os.path.basename(args.quantized)}:")
        print(f"  Avg MSE: {sum(mses)/len(mses):.8f} (over {len(mses)} total layers)")
    else:
        print("  Error: No MSE data could be calculated.")

if __name__ == '__main__':
    main()
"""

with open(eval_path, 'w') as f: f.write(full_script)

print("--- FULL EVALUATION (ALL LAYERS) ---")
print("\\n--- 60% THRESHOLD ---")
!python scripts/eval_reconstruction.py --model-dir /content/models/gemma-4-E2B --quantized quantized/gemma_svd_60.pt

print("\\n--- 70% THRESHOLD ---")
!python scripts/eval_reconstruction.py --model-dir /content/models/gemma-4-E2B --quantized quantized/gemma_svd_70.pt

--- FULL EVALUATION (ALL LAYERS) ---
\n--- 60% THRESHOLD ---

Results for gemma_svd_60.pt:
  Avg MSE: 0.01187033 (over 182 total layers)
\n--- 70% THRESHOLD ---
